In [0]:
current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

silver_scd1_path = f"{base_path}/silver/earthquakes_scd1"

# ==============================================================================
# SECTION 1: Liquid Clustering Configuration
# ==============================================================================
# Apply Liquid Clustering on event_id and event_time for optimal lookup and range pruning
spark.sql(f"""
    ALTER TABLE delta.`{silver_scd1_path}` 
    CLUSTER BY (event_id, event_time)
""")
# print("[LIQUID CLUSTERING] Applied CLUSTER BY (event_id, event_time).")

# ==============================================================================
# SECTION 2: Idempotent File Compaction (OPTIMIZE)
# ==============================================================================
# Compact small micro-batch files using Liquid Clustering key layout
spark.sql(f"OPTIMIZE delta.`{silver_scd1_path}`")
# print("[OPTIMIZE SUCCESS] Bin-packed small Parquet files according to cluster keys.")

# ==============================================================================
# SECTION 3: Garbage Collection & Vacuuming (Serverless Safe)
# ==============================================================================
# Execute standard VACUUM using default governance retention window
spark.sql(f"VACUUM delta.`{silver_scd1_path}`")
# print("[VACUUM SUCCESS] Cleaned stale tombstoned files outside retention window.")

In [0]:
# current_user = spark.sql("SELECT current_user()").collect()[0][0]
# default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

# dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
# base_path = dbutils.widgets.get("base_path")

# silver_scd1_path = f"{base_path}/silver/earthquakes_scd1"

# df_detail = spark.sql(f"DESCRIBE DETAIL delta.`{silver_scd1_path}`")
# df_history = spark.sql(f"DESCRIBE HISTORY delta.`{silver_scd1_path}`")

# print("=== TABLE STORAGE & CLUSTERING DETAILS ===")
# df_detail.select("format", "numFiles", "sizeInBytes", "clusteringColumns").show(truncate=False)

# print("\n=== RECENT TRANSACTION HISTORY ===")
# df_history.select("version", "timestamp", "operation", "operationParameters").show(5, truncate=False)